# RL fine-tuning of a generative recommender against a learned reward model

A GPT-2 decoder over item identifiers is trained to predict a user's next item and is then fine-tuned with
reinforcement learning to generate recommendation lists with a higher predicted user response. The response is
predicted by a second decoder, the reward model. The study measures what the fine-tuning changes: the predicted
response, the relevance of the lists, and how much the lists of different users differ.

**Notation.** A user's log is a sequence of events $(i_t, \rho_t, c_t, u_t)$: item $i_t$, played ratio
$\rho_t \in [0, 1]$, session continuation $c_t \in \{0, 1\}$ and return $u_t \in \{0, 1\}$. The recommender is
$\pi_\theta$, and $\pi_0$ is its state before fine-tuning. A recommendation list $a_{1:K}$ is generated item by
item from a history $h$, and $F$ is the set of the next $M$ items the user consumed after $h$.

## Setup

In [ ]:
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

MAX_MOUNT_DEPTH = 4
manifests = [
    manifest_path
    for depth in range(1, MAX_MOUNT_DEPTH + 1)
    for manifest_path in sorted(Path("/kaggle/input").glob("*/" * depth + "gptrec_bundle/manifest.json"))
]
assert manifests, "gptrec_bundle not found under /kaggle/input"
BUNDLE_DIR = manifests[0].parent
manifest = json.loads((BUNDLE_DIR / "manifest.json").read_text())
print("bundle:", BUNDLE_DIR)
for dataset, result in manifest["smoke_tests"].items():
    print(f"smoke test on {dataset}:", result.splitlines()[0])

WORKING = Path("/kaggle/working")
SOURCE_DIR = WORKING / "src"
shutil.copytree(BUNDLE_DIR / "src", SOURCE_DIR, dirs_exist_ok=True)
os.environ.update(GPTREC_HOME=str(WORKING), HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
sys.path.insert(0, str(SOURCE_DIR))

missing = [package for package in ['torch', 'transformers', 'polars', 'pandas', 'numpy', 'huggingface_hub', 'tqdm', 'matplotlib'] if importlib.util.find_spec(package) is None]
if missing:
    wheels = BUNDLE_DIR / "wheels"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-index", f"--find-links={wheels}", *missing])

In [ ]:
import os

import polars
import torch
import transformers

print(f"torch {torch.__version__} | transformers {transformers.__version__} | polars {polars.__version__} | {os.cpu_count()} CPU cores")
assert torch.cuda.is_available(), "no CUDA device"
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {properties.name}, {properties.total_memory / 2**30:.0f} GB, compute capability {torch.cuda.get_device_capability(index)}")

device = torch.device("cuda:0")
assert torch.isfinite((torch.randn(64, 64, device=device) @ torch.randn(64, 64, device=device)).sum()).item()
torch.set_float32_matmul_precision("high")

In [ ]:
import shutil

import pandas as pd

from gptrec.baselines.suite import BaselineConfig
from gptrec.data.datasets import download_raw_data, load_raw_events
from gptrec.data.preprocess import PreprocessingConfig
from gptrec.experiment import DEFAULT_REWARDS, Experiment, ExperimentConfig, run_smoke_test
from gptrec.models.backbone import TransformerConfig
from gptrec.rollout import GenerationConfig
from gptrec.training.rl import RLConfig
from gptrec.training.supervised import OptimizationConfig

SECONDS_PER_DAY = 86_400


def show(table):
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.float_format", "{:.4f}".format):
        display(table)

## Configuration

In [ ]:
DATASET = "kion"

CONFIGS = {
    "yambda": ExperimentConfig(
        preprocessing=PreprocessingConfig(max_items=50_000, min_item_events=20, min_user_events=50, validation_days=7, test_days=7),
        transformer=TransformerConfig(hidden_size=256, num_layers=4, num_heads=4, dropout=0.1, max_positions=128),
        window_stride=32,
        policy_optimization=OptimizationConfig(epochs=10, batch_size=256, learning_rate=1e-3),
        reward_model_optimization=OptimizationConfig(epochs=8, batch_size=512, learning_rate=1e-3),
        baselines=BaselineConfig(optimization=OptimizationConfig(epochs=10, batch_size=256, learning_rate=1e-3)),
        horizon=10,
        max_future=20,
        ks=(1, 5, 10),
        evaluation_batch_size=1024,
        rl=RLConfig(num_steps=2000, histories_per_step=512, reference_kl_weight=0.3, learning_rate=1e-5, evaluate_every=250, mixed_precision=True),
        sweep_steps=1000,
        kl_weights=(0.0, 0.01, 0.03, 0.1, 0.3, 1.0),
        rewards=DEFAULT_REWARDS[3:5],
        reward_kl_weights=(0.1,),
        choice_reward_kl_weights=(),
        logged_relevance_weights=(10.0,),
        logged_relevance_kl_weights=(0.0, 0.03),
        personal_reward_kl_weights=(0.0, 0.03),
        credit_ablation_kl_weights=(0.03,),
        importance_weight_caps=(0.5,),
        sampled_list_counts=(8,),
    ),
    "kion": ExperimentConfig(
        preprocessing=PreprocessingConfig(max_items=20_000, min_item_events=10, min_user_events=5, validation_days=7, test_days=7,
                                          session_gap_seconds=0, return_window_seconds=7 * SECONDS_PER_DAY),
        transformer=TransformerConfig(hidden_size=256, num_layers=4, num_heads=4, dropout=0.2, max_positions=48),
        window_stride=24,
        policy_optimization=OptimizationConfig(epochs=30, batch_size=512, learning_rate=1e-3),
        reward_model_optimization=OptimizationConfig(epochs=15, batch_size=512, learning_rate=1e-3),
        baselines=BaselineConfig(optimization=OptimizationConfig(epochs=30, batch_size=512, learning_rate=1e-3)),
        horizon=10,
        max_future=20,
        ks=(1, 5, 10),
        evaluation_batch_size=2048,
        rl=RLConfig(num_steps=2000, histories_per_step=512, min_history=3, generation=GenerationConfig(block_history_items=True),
                    reference_kl_weight=0.3, learning_rate=1e-5, evaluate_every=250, mixed_precision=True),
        sweep_steps=1000,
        kl_weights=(0.0, 0.1, 0.2, 0.3, 0.5, 1.0),
        rewards=DEFAULT_REWARDS[3:5],
        reward_kl_weights=(0.1, 0.3),
        choice_reward_kl_weights=(),
        logged_relevance_weights=(10.0, 3.0),
        logged_relevance_kl_weights=(0.0, 0.03, 0.1, 0.3),
        personal_reward_kl_weights=(0.0, 0.1, 0.3),
        credit_ablation_kl_weights=(0.1, 0.3),
        importance_weight_caps=(0.5, 2.0),
        sampled_list_counts=(8,),
    ),
}
SMOKE_TEST_USERS = {"yambda": 200, "kion": 30_000}

In [ ]:
config = CONFIGS[DATASET]
HOME = WORKING / DATASET
TIME_BUDGET_HOURS = 11.0

In [ ]:
raw_events = load_raw_events(DATASET, BUNDLE_DIR / manifest["raw_data"][DATASET])
print(f"{len(raw_events):,} raw events")

## Smoke test

All stages on a subsample of users, with small models and a few optimisation steps.

In [ ]:
run_smoke_test(raw_events, config, WORKING / "smoke_test" / DATASET, device, max_users=SMOKE_TEST_USERS[DATASET])
shutil.rmtree(WORKING / "smoke_test")

## 1. Data

Events are split on one time axis: the last $T_{\text{test}}$ days form the test period, the $T_{\text{val}}$ days
before them the validation period, everything earlier the training period. The catalogue consists of the most
frequent items of the training period, and item statistics are computed on the training period only.

Two retention signals are defined for an event at time $t$ from the times $t' \ge t$ of the same user's later events:

$$c = \mathbb{1}\big[\min_{t'}\,(t' - t) \le g\big], \qquad u = \mathbb{1}\big[\exists\, t' :\; g < t' - t \le w\big],$$

with session gap $g$ and return window $w$. An outcome counts as observed only if its waiting time ends inside the
observed data: $t + g \le T$ for $c$ and $t + w \le T$ for $u$, where $T$ is the end of the training period for
training events and the end of the log for held-out events. Unobserved outcomes enter neither training nor
evaluation.

In [ ]:
experiment = Experiment(config, HOME, device, time_budget_hours=TIME_BUDGET_HOURS)
summary = experiment.prepare_data(raw_events)
del raw_events
show(summary)

## 2. Base recommender

A causal transformer (GPT-2) with tied input and output item embeddings $E$ models the item sequence of a user:

$$\pi_\theta(i_{t+1} = j \mid i_{1:t}) = \operatorname{softmax}_j\,(E\, s_t), \qquad
\mathcal{L}_{\text{next}}(\theta) = -\sum_t \log \pi_\theta(i_{t+1} \mid i_{1:t}),$$

where $s_t$ is the state of the transformer after $t$ items. A list of $K$ items is produced in two ways:

- **top-K**: the $K$ most probable next items under $\pi_\theta(\cdot \mid h)$;
- **next-K**: item by item, $a_k = \arg\max_j \pi_\theta(j \mid h, a_{1:k-1})$, without repeating an item.

Relevance is measured on one request per test user, issued at the start of the test period, against the user's
next $M$ events: all of them, those with $\rho \ge 0.5$ (*liked*) and those with $\rho < 0.5$ (*skipped*). A second
split compares the played ratio with the mean $\bar\rho(i)$ of the same item in the training period: $\rho \ge \bar\rho(i)$
(*above usual*) and $\rho < \bar\rho(i)$ (*below usual*); unlike the first, it does not favour items that are easy to
finish. The metrics are Recall@K, NDCG@K, MAP@K and HitRate@K.

In [ ]:
show(experiment.pretrain_policy())

## 3. Reward model

A second causal transformer reads, at position $t$, the item $i_t$ together with the quantised played ratio
$\rho_{t-1}$ of the previous item. Its state $z_t$ depends on $(i_{1:t}, \rho_{1:t-1})$: it has seen the candidate
item and the history, but not the response to the candidate. Four heads predict that response,

$$(\hat\rho_t,\ \hat c_t,\ \hat u_t,\ \hat q_t) = \sigma(W z_t),$$

the played ratio, the session continuation, the return, and the probability that the user chooses the item at
all. The loss is

$$\mathcal{L} = \operatorname{BCE}(\hat\rho, \rho) + \lambda_c \operatorname{BCE}(\hat c, c) + \lambda_u \operatorname{BCE}(\hat u, u) + \lambda_q\, \mathcal{L}_q,$$

with $\rho$ as a soft label and the retention terms taken over observed outcomes only. Logged events contain only
chosen items, so $\hat q$ is learned by contrast. One event per sequence is scored a second time with its item
replaced by the item $j$ of another event of the batch, so that replacements follow the empirical item
distribution $p$:

$$\mathcal{L}_q = -\tfrac{1}{2}\, \mathbb{E}\big[\log \hat q(i_t \mid h_t)\big] - \tfrac{1}{2}\, \mathbb{E}_{j \sim p}\big[\log\big(1 - \hat q(j \mid h_t)\big)\big],
\qquad \hat q^{*}(j \mid h) = \frac{P(j \mid h)}{P(j \mid h) + p(j)}.$$

Two copies are trained on disjoint halves of the users. $R_{\text{train}}$ provides the rewards during fine-tuning;
$R_{\text{held-out}}$ only evaluates the resulting lists.

The table compares the predicted played ratio on test-period events, each predicted from the events before it,
with three reference predictors: the training mean, the mean of the item, and the mean over the user's preceding
events.

In [ ]:
show(experiment.train_reward_models())

**Error structure.** Quality of $\hat\rho$ separately for events the user initiated and events a recommender served;
mean signed error $\hat\rho - \rho$ by decile of item popularity and of item duration; correlation between the
predictions of the two copies on the same events.

In [ ]:
for report in ["reward_model_quality_by_origin", "reward_model_error_by_popularity_decile", "reward_model_error_by_duration_decile", "reward_model_agreement"]:
    print(report)
    show(pd.read_csv(experiment.reports.directory / f"{report}.csv", index_col=0))

**What each signal offers as a reward.**

- *Retention*: prediction quality of $\hat c$ and $\hat u$ on events with observed outcomes.
- *Choice*: AUC of $\hat q$ for the logged item against a replaced one.
- *Spread over candidates*: for the $N$ most probable next items of each test user under $\pi_0$, the standard
  deviation of each predicted signal across the candidates of one user, averaged over users; the standard deviation
  of its per-user mean across users; and the AUC of the signal for the event that the candidate is in $F$. The last
  row gives the same for $\log \pi_0$.

In [ ]:
for report, index_columns in [("reward_model_retention", [0, 1, 2]), ("reward_model_choice", 0), ("reward_signal_spread", 0)]:
    print(report)
    show(pd.read_csv(experiment.reports.directory / f"{report}.csv", index_col=index_columns))

## 4. Baselines

| Model | Definition |
|---|---|
| Most popular | the most frequent items of the training period |
| Replay history | the items of the user's history, most played first; the rest by popularity |
| Markov chain | first order, $P(j \mid i) \propto n(i \to j)$ |
| GRU4Rec [5] | GRU over the item sequence, full-softmax cross-entropy |
| SASRec [6] | causal transformer, binary cross-entropy against one sampled negative per position |
| BERT4Rec [7] | bidirectional transformer trained on masked-item prediction; a mask is appended at inference |

The neural baselines share the width, depth, dropout, sequence length and training windows of the recommender.
Every model ranks the catalogue for the same test requests under the same item constraints.

In [ ]:
show(experiment.compare_baselines())

## 5. Reinforcement-learning fine-tuning

Generating a list is an episode of $K$ steps with state $s_k = (h, a_{1:k-1})$ and action
$a_k \sim \pi_\theta(\cdot \mid s_k)$. The reward is the response $R_{\text{train}}$ predicts to $a_k$; for the items
generated before $a_k$ the reward model is given its own expected played ratio as their feedback:

$$r_k = \hat\rho\,(a_k \mid h, a_{1:k-1}), \qquad G_k = \sum_{l \ge k} \gamma^{\,l-k}\, r_l .$$

A value model $V_\phi$, initialised from the recommender, regresses on $G_k$ and yields the advantages by
generalised advantage estimation [4],

$$\delta_k = r_k + \gamma V_\phi(s_{k+1}) - V_\phi(s_k), \qquad A_k = \sum_{l \ge 0} (\gamma\lambda)^l\, \delta_{k+l},$$

which are standardised within each batch. The policy minimises the clipped surrogate of PPO [3] with a KL penalty
towards the base recommender:

$$\mathcal{L}(\theta) = -\,\mathbb{E}\Big[\min\big(w_k A_k,\ \operatorname{clip}(w_k, 1 - \epsilon, 1 + \epsilon)\, A_k\big)\Big]
+ \beta\, \mathbb{E}\Big[\operatorname{KL}\big(\pi_\theta(\cdot \mid s_k)\,\|\,\pi_0(\cdot \mid s_k)\big)\Big],
\qquad w_k = \frac{\pi_\theta(a_k \mid s_k)}{\pi_{\theta_{\text{old}}}(a_k \mid s_k)} .$$

The reward contains no relevance term, so $\beta$ is the only constraint that ties the policy to observed
behaviour.

Fine-tuned lists are compared with those of $\pi_0$ on the test requests by relevance, by the response both reward
models predict, and by concentration: catalogue coverage, the Gini coefficient of item exposure, the share of users
who receive the most frequent item, and the expected overlap between the lists of two users.

In [ ]:
show(experiment.finetune())

Per-user change from $\pi_0$ to $\pi_\theta$ with a 95% paired-bootstrap interval over users.

In [ ]:
show(pd.read_csv(experiment.reports.directory / "rl_vs_pretrained_significance.csv", index_col=0))

The exposure of an item is the number of test users whose list contains it. The items with the largest gain and
the largest loss from $\pi_0$ to $\pi_\theta$:

In [ ]:
exposure = pd.read_csv(experiment.reports.directory / "rl_exposure_change.csv", index_col=0)
show(exposure.head(15))
show(exposure.tail(15))

## 6. Relevance against reward

The fine-tuning is repeated for several values of $\beta$. Each run is a point: the predicted played ratio of its
lists against their NDCG@K. The ratio is predicted by $R_{\text{train}}$ and by $R_{\text{held-out}}$; a gap between
the two is a gain that only the optimised model predicts.

The second figure places the same runs by the divergence they ended at,
$\mathbb{E}\big[\operatorname{KL}(\pi_\theta(\cdot \mid s_k)\,\|\,\pi_0(\cdot \mid s_k))\big]$ in nats per generated item. The
weight $\beta$ acts relative to the standardised advantages, so its effect depends on the estimator and the data;
the divergence reached does not.

In [ ]:
show(experiment.sweep_kl_weight())

## 7. Whose reward is it

A list $a$ generated for a user with history $h$ can also be shown to another user with history $h'$. The reward
model then gives two values,

$$R_{\text{own}} = \mathbb{E}\big[\hat\rho(a \mid h)\big], \qquad R_{\text{other}} = \mathbb{E}\big[\hat\rho(a \mid h')\big],$$

and $R_{\text{own}} - R_{\text{other}}$ is the part of the predicted response that depends on who receives the list; it
is zero for any recommender that ignores the user. The gain of a method over $\pi_0$ splits accordingly into the gain
that other users would give the same lists, and the rest, its personal share:

$$\Delta R_{\text{own}} = \Delta R_{\text{other}} + \Delta\,(R_{\text{own}} - R_{\text{other}}).$$

As a reference that uses no reward model, the $N$ candidates of $\pi_0$ are re-ranked by
$\bar\rho(j) + w \log \pi_0(j \mid h)$, where $\bar\rho(j)$ is the mean played ratio of item $j$ in the training period.

In [ ]:
show(experiment.explain_reward_gain())

## 8. Alternatives to on-policy RL

**Liked-only fine-tuning.** Next-item training restricted to targets the user liked:
$\mathcal{L}(\theta) = -\sum_{t:\ \rho_{t+1} \ge 0.5} \log \pi_\theta(i_{t+1} \mid i_{1:t})$.

**Off-policy REINFORCE on logged sessions** [2]. Logged items are the actions and $\pi_0$ is the behaviour policy:

$$\mathcal{L}(\theta) = -\frac{\sum_t \gamma^{\tau_t}\, \bar\omega_t\, G_t\, \log \pi_\theta(i_t \mid i_{1:t-1})}{\sum_t \gamma^{\tau_t}\, \bar\omega_t},
\qquad \bar\omega_t = \min\Big(\kappa,\ \prod_{l=t}^{t+L-1} \frac{\pi_\theta(i_l \mid i_{1:l-1})}{\pi_0(i_l \mid i_{1:l-1})}\Big),$$

where $G_t$ is the discounted sum of logged rewards from $t$ to the end of its session, $\tau_t$ is the position of
$t$ in the session, and the product runs over at most $L$ steps of the same session. The cap $\kappa$ bounds the
distance from $\pi_0$.

**Re-ranking.** The $N$ most probable next items under $\pi_0$ are scored one by one,
$\operatorname{score}(j) = r(j \mid h) + w \log \pi_0(j \mid h)$, and the $K$ best form the list.

**Best of $n$ sampled lists**, the test-time scaling of [2]. $n$ lists are sampled from $\pi_0$, restricted at each
step to its $m$ most probable items, and the list with the largest $\sum_k \gamma^{k-1} r_k$ is served.

Every method is placed by the response $R_{\text{held-out}}$ predicts to its lists, $R_{\text{own}}$ in the first figure
and $R_{\text{own}} - R_{\text{other}}$ in the second, against NDCG@K on all and on liked items and against catalogue
coverage.

In [ ]:
show(experiment.compare_alternatives_to_rl())

## 9. Rewards for continuation and return

The reward generalises to

$$r_k = w_\rho\,\big(\hat\rho_k - b_k\big) + w_c\, \hat c_k + w_u\, \hat u_k + w_F\, \frac{\mathbb{1}[a_k \in F]}{\log_2(k + 1)},$$

with a baseline $b_k$ that is zero unless stated otherwise.

| Reward | $w_\rho$ | $w_c$ | $w_u$ |
|---|---|---|---|
| played ratio | 1 | 0 | 0 |
| session | 0 | 1 | 0 |
| return | 0 | 0 | 1 |
| ratio + return | 1 | 0 | 1 |
| ratio + session + return | 1 | 1 | 1 |

*Played time* replaces $\hat\rho_k$ by $\hat\rho_k\, d(a_k) / \operatorname{median}(d)$, where $d$ is the duration of
an item. The rewards selected in the configuration are run for several $\beta$. The figures show NDCG@K and catalogue
coverage against one signal predicted by $R_{\text{held-out}}$, one line per reward.

In [ ]:
show(experiment.sweep_rewards())

## 10. Relevance inside the reward

**Logged relevance**, $r_k = \mathbb{1}[a_k \in F] / \log_2(k + 1)$, is the term of DCG@K, the NDCG reward of [1]. It is
run alone with $\beta = 0$, and added to the played ratio,

$$r_k = \hat\rho_k + w_F\, \frac{\mathbb{1}[a_k \in F]}{\log_2(k + 1)},$$

for several $w_F$ and several $\beta$. The two constraints protect different things: the reward term protects the hits
on the user's real continuation, the KL penalty protects the whole distribution of $\pi_0$, including its variety.

In [ ]:
show(experiment.sweep_relevance_in_reward())

## 11. Personal rewards

A reward without a baseline pays the most for items that everybody finishes, whoever receives them. Two rewards pay
only for what depends on the user:

$$r_k = \hat\rho(a_k \mid h, a_{1:k-1}) - \hat\rho(a_k \mid h', a_{1:k-1}) \quad \text{(personal ratio)},
\qquad r_k = \hat\rho(a_k \mid h, a_{1:k-1}) - \bar\rho(a_k) \quad \text{(ratio above item mean)},$$

where $h'$ is the history of another user of the batch. For a policy that ignores the user the first has zero
expectation, $\mathbb{E}_{h, h'}\big[\hat\rho(a \mid h) - \hat\rho(a \mid h')\big] = 0$, so a list that is the same for
everyone earns nothing; it also cancels every error of the reward model that depends on the item alone. Both rewards
are run for several $\beta$. Re-ranking the candidates of $\pi_0$ by the personal ratio gives the same reward without RL.

In [ ]:
show(experiment.sweep_personal_rewards())

## 12. Credit along the list

Two ablations test whether fine-tuning uses the sequential structure of a list. With $\gamma = 0$ the return reduces to
the reward of the item itself, $G_k = r_k$, so no item is credited with what follows it. With a horizon of one, a
single item is sampled per episode during fine-tuning, so the policy never learns from a context that contains
generated items; its lists are still generated item by item at evaluation. Both are compared with the runs of
section 6 at the same $\beta$.

In [ ]:
show(experiment.ablate_sequence_credit())

## 13. Multi-step generation

For each list position $k$: the probability that $a_k \in F$, the entropy of $\pi_\theta(\cdot \mid s_k)$, the
predicted signals, and the disagreement $|\hat\rho^{\,\text{train}}_k - \hat\rho^{\,\text{held-out}}_k|$ between the two
reward models.

A generated item has no observed feedback, but the reward model needs one to score the next item. Three
substitutes are compared: the expected ratio $\hat\rho_k$, a sample from $\operatorname{Bernoulli}(\hat\rho_k)$, and $1$.

The table gives NDCG@k of top-K and next-K lists before and after fine-tuning.

In [ ]:
show(experiment.analyse_multistep())

## Reports

In [ ]:
shutil.make_archive(str(WORKING / f"{DATASET}_reports"), "zip", experiment.reports.directory)
sorted(path.name for path in experiment.reports.directory.iterdir())

## References

1. A. V. Petrov, C. Macdonald. Aligning GPTRec with Beyond-Accuracy Goals with Reinforcement Learning. arXiv:2403.04875.
2. A. Matveev, S. Makeev, A. Krasilnikov, V. Baikalov, S. Liamaev, K. Khrylchenko. Session-Level Optimization for
   Large-Scale Retrieval using REINFORCE with Multi-Step Off-Policy Correction. arXiv:2607.02818.
3. J. Schulman, F. Wolski, P. Dhariwal, A. Radford, O. Klimov. Proximal Policy Optimization Algorithms. arXiv:1707.06347.
4. J. Schulman, P. Moritz, S. Levine, M. Jordan, P. Abbeel. High-Dimensional Continuous Control Using Generalized
   Advantage Estimation. arXiv:1506.02438.
5. B. Hidasi, A. Karatzoglou, L. Baltrunas, D. Tikk. Session-based Recommendations with Recurrent Neural Networks.
   arXiv:1511.06939.
6. W.-C. Kang, J. McAuley. Self-Attentive Sequential Recommendation. arXiv:1808.09781.
7. F. Sun et al. BERT4Rec: Sequential Recommendation with Bidirectional Encoder Representations from Transformer.
   arXiv:1904.06690.
8. A. Ploshkin et al. Yambda-5B: A Large-Scale Multi-modal Dataset for Ranking and Retrieval. arXiv:2505.22238.
9. A. Petrov, I. Safilo, D. Tikhonovich, D. Ignatov. MTS Kion Implicit Contextualised Sequential Dataset for Movie
   Recommendation. arXiv:2209.00325.